<a href="https://colab.research.google.com/github/AnaraHayat/flyrank_assignment1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item**, identified by `content_id`, pseudonymized under one
of 32 `client_id` values.

**Time window:** every metric column is a trailing 90-day total ending at export time
(`impressions_90d`, `clicks_90d`, `sessions_90d`, etc.). Two shorter windows sit inside that
90 days for trend comparison: `*_last_30d` (most recent 30 days) and `*_prev_30d` (the 30 days
before that, days 31-60 back). `trend_direction` and `trend_pct` compare these two 30-day windows
against each other, not against the full 90 days.

Every page in this slice is already at least 90 days old (`content_age_days` starts at 90) — this
teaching slice does not contain brand-new content.

In [6]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("rows, columns:", df.shape)

# grain check: content_id should be unique per row
dup = df.groupby("content_id").size()
print("content_ids that appear more than once:", (dup > 1).sum(), "of", len(dup))

# time window check
print("content_age_days min/max:", df["content_age_days"].min(), df["content_age_days"].max())
print("age_tier values in this slice:", sorted(df["age_tier"].unique()))


rows, columns: (30000, 44)
content_ids that appear more than once: 0 of 30000
content_age_days min/max: 90 564
age_tier values in this slice: ['181-365', '31-90', '365+', '91-180']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** — knowable at prediction time, safe for the model:
`search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`,
`word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `impressions_90d`,
`clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`,
`ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`,
`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`.

**Label / proxy** — the thing being predicted, never a feature:
`trend_direction`, `trend_pct` → `is_declining_label = (trend_direction == "down")`.

**Context** — for grouping, joining, splitting only, never for the model to learn from:
`content_id` (unique per row, joins), `client_id` (32 values, used for client-holdout splits
so the model is tested on clients it never trained on).

**Excluded** — each with a reason:
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`,
  `clicks_prev_30d`, `sessions_prev_30d` — these are the exact inputs `trend_pct` and
  `trend_direction` are computed from. Using them as features would leak the label straight
  back in.
- `provider_used`, `model_used` — which LLM generated or touched the article. Not a signal
  about search performance, and the data dictionary flags both as "not a model feature".
- `age_tier`, `age_tier_order`, `freshness_tier`, `word_count_tier`, `char_count_tier`,
  `impression_tier`, `position_tier` — bucketed (or bucket-ordered) versions of numeric
  columns already kept as features. Keeping both the raw number and its bucket is redundant,
  so the raw column stays and the tier is dropped.

In [7]:
feature_cols = [
    "search_volume", "competition", "competition_level", "cpc", "content_type", "main_intent",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
label_cols = ["trend_direction", "trend_pct"]
context_cols = ["content_id", "client_id"]
excluded_cols = [
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "provider_used", "model_used",
    "age_tier", "age_tier_order", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier",
]

all_buckets = feature_cols + label_cols + context_cols + excluded_cols
print("columns sorted:", len(all_buckets), "/ columns in file:", df.shape[1])
print("missing from my buckets:", set(df.columns) - set(all_buckets))
print("any column in more than one bucket?:", len(all_buckets) != len(set(all_buckets)))

columns sorted: 44 / columns in file: 44
missing from my buckets: set()
any column in more than one bucket?: False


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# Grain: one row per content_id — zero rows back means the grain holds
grain_check = df.groupby("content_id").size()
print("content_ids with count > 1:", (grain_check > 1).sum())

# Counts: total rows, distinct clients, rows per client
print("total rows:", len(df))
print("distinct clients:", df["client_id"].nunique())
print(df.groupby("client_id").size().describe()[["min", "25%", "50%", "75%", "max"]])


content_ids with count > 1: 0
total rows: 30000
distinct clients: 32
min       3.00
25%     110.25
50%     567.00
75%    1058.75
max    7008.00
dtype: float64


In [9]:
# Missingness: overall, then grouped by content_type to check the "systematic, not random" claim
check_cols = ["search_volume", "competition", "cpc", "main_intent", "word_count", "char_count"]
print("overall missing rate:")
print(df[check_cols].isna().mean().round(3))

print()
print("missing rate by content_type (search_volume, word_count):")
print(df.groupby("content_type")[["search_volume", "word_count"]].apply(lambda g: g.isna().mean()).round(3))


overall missing rate:
search_volume    0.082
competition      0.082
cpc              0.082
main_intent      0.079
word_count       0.257
char_count       0.257
dtype: float64

missing rate by content_type (search_volume, word_count):
                    search_volume  word_count
content_type                                 
comparison article          0.000       0.000
feedly article              1.000       0.000
keyword article             0.014       0.283


In [10]:
# Windows: content_age_days range, and the two flagged "0 means no data, not literally 0" columns
print("content_age_days: min", df["content_age_days"].min(), "max", df["content_age_days"].max())
print("avg_position == 0 (no position data, not position zero):", (df["avg_position"] == 0).sum(), "rows")
print("impressions_prev_30d == 0 (trend_pct filled with 0 here):", (df["impressions_prev_30d"] == 0).sum(), "rows")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print()
print("trend_direction breakdown:")
print(df["trend_direction"].value_counts())
print("is_declining_label rate:", round(df["is_declining_label"].mean(), 3))


content_age_days: min 90 max 564
avg_position == 0 (no position data, not position zero): 1205 rows
impressions_prev_30d == 0 (trend_pct filled with 0 here): 3388 rows

trend_direction breakdown:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
is_declining_label rate: 0.542


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Missingness is patterned, not random.** `search_volume` is missing for 100% of `feedly
  article` rows and near-0% of `keyword article` rows (verified above). A blind
  `fillna(0)` on `search_volume` would silently tell the model "this is a feedly article" —
  the model would be learning content type through the back door, not a real search signal.
- **`avg_position == 0` means "no position data recorded", not literally rank zero.** 1,205
  rows carry this. Treating it as a real position would make those pages look like impossibly
  strong performers.
- **This is one 90-day snapshot per page, not a real time series.** There are only two
  comparison points inside it (last 30 days vs. the 30 before). It can say a page trended down
  over that window — it cannot say what happens next month, or show a smooth trajectory.
- **Client coverage is very unbalanced.** 32 clients, but page counts per client range from 3
  to 7,008 (median 567). A model trained without client-aware splitting could just be memorizing
  a few large clients' quirks — this is exactly why `client_id` stays in "context" for
  client-holdout splits, never as a feature.
- **Every page here is already ≥ 90 days old.** The data can say nothing about how brand-new
  content performs or trends — that population isn't in this slice at all.
- **Correlational, not causal.** Nothing here proves *why* a page is declining — only that it
  is, by this window's definition. Confounders (a client-wide algorithm hit, a seasonal dip)
  aren't visible in page-level columns alone.

In [11]:
# back up the two limits above with numbers
print("feedly article rows missing search_volume:",
      df.loc[df.content_type == "feedly article", "search_volume"].isna().mean())
print("keyword article rows missing search_volume:",
      df.loc[df.content_type == "keyword article", "search_volume"].isna().mean())

per_client = df.groupby("client_id").size()
print("rows per client -> min:", per_client.min(), "max:", per_client.max(), "median:", per_client.median())


feedly article rows missing search_volume: 1.0
keyword article rows missing search_volume: 0.01367295181387143
rows per client -> min: 3 max: 7008 median: 567.0


## Self-check

Before you submit, confirm each line honestly:

- [Done] Every section above is filled — markdown thinking AND the code that backs it
- [Done ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Done ] No client names, URLs, or private queries anywhere
- [Done ] My claims use careful words: observed, measured, directional, decision-support
- [Done ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.